*0.2 Math / ML basics*

# Train/val/test split

**The situation.** A classifier reaches 99% accuracy in the notebook. It ships. In production it gets 71%. The 99% was measured on the same tickets it learned from — it had memorised them, and nobody had held anything back to find out.

**Three piles.** *Train*: the model learns from these. *Validation*: you use these to choose settings and decide when to stop — the model never learns from them, but you look at them many times. *Test*: touched once, at the end, to report the number you will tell others. Once you tune on the test set, it has become a validation set and you need a new one.

In [1]:
# Load OPENAI_API_KEY from the .env file. The OpenAI clients read it from the environment.
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())
MODEL = "gpt-4o-mini"

**Split 120 support messages, keeping the category mix the same in every pile.** `stratify` does that; without it, a small category can end up missing from the test set entirely.

In [2]:
from collections import Counter

from sklearn.model_selection import train_test_split

templates = {
    "billing": [
        "charged twice for {x}",
        "refund for {x} not received",
        "invoice for {x} is wrong",
        "cancel subscription {x}",
    ],
    "technical": [
        "{x} page will not load",
        "error 500 when opening {x}",
        "app crashes on {x}",
        "cannot upload {x}",
    ],
    "account": [
        "reset password for {x}",
        "change email on {x}",
        "delete my {x} account",
        "two-factor code for {x} missing",
    ],
}
texts = []
labels = []
for label, patterns in templates.items():
    for pattern in patterns:
        for x in (
            "order",
            "the dashboard",
            "my plan",
            "the mobile app",
            "the report",
            "the team workspace",
            "billing",
            "settings",
            "the export",
            "the API",
        ):
            texts.append(pattern.format(x=x))
            labels.append(label)

train_texts, rest_texts, train_labels, rest_labels = train_test_split(
    texts, labels, test_size=0.4, stratify=labels, random_state=0
)
val_texts, test_texts, val_labels, test_labels = train_test_split(
    rest_texts, rest_labels, test_size=0.5, stratify=rest_labels, random_state=0
)

for name, subset in (("train", train_labels), ("val", val_labels), ("test", test_labels)):
    print(f"{name:<6} {len(subset):>3} messages  {dict(Counter(subset))}")
assert len(train_texts) == 72 and len(val_texts) == 24 and len(test_texts) == 24
assert not set(train_texts) & set(test_texts)

train   72 messages  {'billing': 24, 'account': 24, 'technical': 24}
val     24 messages  {'technical': 8, 'billing': 8, 'account': 8}
test    24 messages  {'technical': 8, 'account': 8, 'billing': 8}


**Reading the output.** 60% train, 20% validation, 20% test, and every pile has the same 1:1:1 category mix. The last assert proves no message appears in two piles.

```
all data  ────▶  train 60%   learn from       (seen many times)
                 val   20%   tune & choose    (looked at many times, never learned from)
                 test  20%   report once      (touched at the very end)
```

**The rule to remember.** Split before you look at anything else. The number you report comes from data the model — and you — never used to make decisions.

| Use it when | Don't when | Instead use |
|---|---|---|
| every model you train or fine-tune; every prompt you tune against examples | data has a time order (predicting next month from this month) — a random split leaks the future | split by time: train on before, test on after |

**Watch out**
- Near-duplicates across piles (the same ticket edited twice) leak just like exact duplicates. De-duplicate first.
- Prompt engineering against the test set is tuning on the test set. Keep a held-out set for prompts too.
- With little data, cross-validation (rotate which fold is validation) gives a steadier number than one split.